<h1 style="color:#56B4E9;">Week 4 - Day 2</h1>

<h2 style="color:#0072B2;">Cross-Validation</h2>

<h3 style="color:#009E73;">Breast Cancer Classification</h3>

This notebook continues the model evaluation work from Day 1 using the same Breast Cancer Wisconsin dataset and the selected KNN model.

The goal today is not to build a new classifier, but to improve how reliably I estimate model performance.

Instead of depending on one validation split, I will use cross-validation to evaluate the model across multiple validation subsets.

<h2 style="color:#0072B2;">Why Is One Validation Split Not Enough?</h2>

A single <span style="color:#E69F00;"><b>validation split</b></span> can give a misleading estimate of model performance, especially when the dataset is relatively small.

The selected validation set may accidentally contain samples that are easier or harder for the trained model than other parts of the dataset.

If I make modeling decisions based only on that score, I may be reacting to the particular split rather than the model's true ability to generalize.

<span style="color:#009E73;"><b>Key idea:</b></span>  
I need an evaluation method that depends less on one particular validation subset.

<h2 style="color:#0072B2;">How Does K-Fold Cross-Validation Solve This Problem?</h2>

In **5-fold cross-validation**, the available training data is divided into five parts.

The model is trained five separate times:

- One fold becomes the <span style="color:#E69F00;"><b>validation fold</b></span>.
- The other four folds become the <span style="color:#0072B2;"><b>training folds</b></span>.
- The validation fold changes in every round.
- Every sample is used for validation once and for training four times.

Instead of depending on one validation score, I can observe the model's performance across several different subsets of the data.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Cross-validation does not guarantee that a model is stable. It allows me to measure how consistent its estimated performance is across different validation subsets.

<h2 style="color:#0072B2;">What Should I Learn From the Five Scores?</h2>

Cross-validation produces several validation scores instead of only one.

- **Mean F1-score** → summarizes the model's average performance across the folds.
- **Standard deviation** → measures how much the F1-scores change from one fold to another.

A high mean is desirable, but it is not enough by itself.

A relatively low standard deviation suggests that the estimated performance is more consistent across different validation subsets.

<span style="color:#009E73;"><b>Key idea:</b></span>  
A useful model evaluation should consider both **performance** and **consistency**, not only the highest score.

In [12]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", data.target_names)

print("\nClass distribution:")
print(y.value_counts())

Dataset shape: (569, 30)
Target shape: (569,)
Classes: ['malignant' 'benign']

Class distribution:
target
1    357
0    212
Name: count, dtype: int64


<h2 style="color:#0072B2;">Why Do I Need Stratification?</h2>

The dataset contains:

- **357 benign samples**
- **212 malignant samples**

The two classes are therefore not equally represented.

If the data is divided without considering class proportions, some folds could contain noticeably different class distributions. This would make their validation scores harder to compare fairly.

For this classification problem, I will use **stratification** to preserve approximately the same class proportions across the different subsets.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Stratification makes the evaluation subsets more comparable by preventing large differences in class balance.

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

Training shape: (455, 30)
Test shape: (114, 30)

Training class distribution:
target
1    285
0    170
Name: count, dtype: int64

Test class distribution:
target
1    72
0    42
Name: count, dtype: int64


<h2 style="color:#0072B2;">Holding Out the Final Test Set</h2>

The dataset was divided into:

- <span style="color:#0072B2;"><b>Training set:</b></span> 455 samples
- <span style="color:#D55E00;"><b>Final test set:</b></span> 114 samples

Stratification preserved similar class distributions in both sets:

- <span style="color:#0072B2;"><b>Training:</b></span> 285 benign and 170 malignant
- <span style="color:#D55E00;"><b>Test:</b></span> 72 benign and 42 malignant

Cross-validation will be performed only on the <span style="color:#0072B2;"><b>training data</b></span>.

The <span style="color:#D55E00;"><b>final test set remains untouched</b></span> during model development.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Cross-validation improves the reliability of model evaluation without sacrificing the independence of the final test set.

In [14]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

for fold, score in enumerate(cv_scores, start=1):
    print(f"Fold {fold}: {score:.4f}")

print(f"\nMean F1: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")

Fold 1: 0.9744
Fold 2: 0.9828
Fold 3: 0.9558
Fold 4: 0.9492
Fold 5: 0.9913

Mean F1: 0.9707
Standard deviation: 0.0159


<h2 style="color:#0072B2;">Cross-Validation Results</h2>

The 5-fold cross-validation produced:

- **Mean F1-score = 0.9707**
- **Standard deviation = 0.0159**
- Fold scores ranging from **0.9492 to 0.9913**

The high mean indicates strong average performance across the five validation folds.

The relatively low standard deviation shows that the estimated performance remained reasonably consistent across the different validation subsets.

<span style="color:#009E73;"><b>Key idea:</b></span>  
The model did not achieve a strong score only on one particular validation subset. Similar performance was observed across all five folds.

In [15]:
day1_validation_f1 = 0.963855421686747

cv_mean_f1 = cv_scores.mean()
difference = cv_mean_f1 - day1_validation_f1

print(f"Day 1 single validation F1: {day1_validation_f1:.4f}")
print(f"Day 2 cross-validation mean F1: {cv_mean_f1:.4f}")
print(f"Difference: {difference:.4f}")

Day 1 single validation F1: 0.9639
Day 2 cross-validation mean F1: 0.9707
Difference: 0.0068


<h2 style="color:#0072B2;">Single Validation Split vs. Cross-Validation</h2>

In Day 1, the selected KNN model with **k = 7** achieved:

<span style="color:#E69F00;"><b>Single Validation F1 = 0.9639</b></span>

Using 5-fold cross-validation on the same training portion of the dataset:

**Cross-Validation F1 = 0.9707 ± 0.0159**

The difference between the single validation score and the cross-validation mean is approximately **0.0068**.

The two estimates are close, suggesting that the Day 1 validation split was reasonably representative.

However, cross-validation provides stronger evidence because the performance estimate is based on multiple validation subsets rather than only one.

<span style="color:#009E73;"><b>Key idea:</b></span>  
The benefit of cross-validation is not simply obtaining a different score. It gives stronger evidence about how reliable the estimated performance is.

In [16]:
for fold, (_, val_index) in enumerate(cv.split(X_train, y_train), start=1):
    y_val_fold = y_train.iloc[val_index]

    counts = y_val_fold.value_counts()
    proportions = y_val_fold.value_counts(normalize=True)

    print(f"Fold {fold}")

    print("Counts:")
    print(counts)

    print("Proportions:")
    print(proportions.round(3))

    print("-" * 30)

Fold 1
Counts:
target
1    57
0    34
Name: count, dtype: int64
Proportions:
target
1    0.626
0    0.374
Name: proportion, dtype: float64
------------------------------
Fold 2
Counts:
target
1    57
0    34
Name: count, dtype: int64
Proportions:
target
1    0.626
0    0.374
Name: proportion, dtype: float64
------------------------------
Fold 3
Counts:
target
1    57
0    34
Name: count, dtype: int64
Proportions:
target
1    0.626
0    0.374
Name: proportion, dtype: float64
------------------------------
Fold 4
Counts:
target
1    57
0    34
Name: count, dtype: int64
Proportions:
target
1    0.626
0    0.374
Name: proportion, dtype: float64
------------------------------
Fold 5
Counts:
target
1    57
0    34
Name: count, dtype: int64
Proportions:
target
1    0.626
0    0.374
Name: proportion, dtype: float64
------------------------------


<h2 style="color:#0072B2;">Did Stratification Preserve the Class Distribution?</h2>

Yes.

Each validation fold contained:

- **57 benign samples → 62.6%**
- **34 malignant samples → 37.4%**

These proportions closely match the class distribution of the training data.

Because the class balance remained consistent across all five folds, differences in F1-score are less likely to be explained by major changes in class proportions.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Stratified K-Fold gives each validation round a comparable class composition, making the fold-to-fold performance comparison more meaningful.

<h2 style="color:#0072B2;">Final Takeaway</h2>

A single <span style="color:#E69F00;"><b>validation split</b></span> can provide a reasonable performance estimate, but its result may still depend on the particular samples selected.

In this experiment, the Day 1 validation F1-score (**0.9639**) was close to the 5-fold cross-validation estimate (**0.9707 ± 0.0159**).

The relatively low variation across folds also showed that the model's estimated performance was consistent across the validation subsets used.

Using **Stratified K-Fold** preserved the class distribution across all folds, making their results more comparable.

<span style="color:#009E73;"><b>Key lesson:</b></span>  
Cross-validation does not simply produce another score. It provides stronger evidence about the reliability and consistency of the model's estimated performance.

The <span style="color:#D55E00;"><b>final test set remained untouched</b></span> throughout this process and should only be used for final evaluation after all model decisions are complete.